In [ ]:
# In this file, we do SFT on Qwen3-0.6B model (Thinking & NoThinking)

In [ ]:
# Fine-tuning Qwen-0.6B-local with Thinking Mode Enabled

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# ====== Base model =======
base_model_path = "Qwen-0.6B-local"
data_path = {
    "train": "validation_QTSA_mix/QTSA_train_thinking.jsonl",
    "validation": "validation_QTSA_mix/QTSA_val_thinking.jsonl"
}
output_dir = "outputs/qwen-0.6B-qtsa-mix-thinking"

# ====== Tokenizer ========
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

max_memory = {0: "30GiB", 1: "30GiB"}

# ====== Load model ========
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    max_memory=max_memory,
    torch_dtype=torch.float16,
    use_cache=False,
    trust_remote_code=True,
)
model.gradient_checkpointing_enable()

# ====== LoRA config ========
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

# ====== Load dataset ========
dataset = load_dataset("json", data_files=data_path)
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

# ====== SFT config ========
sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=50,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir="logs/qwen-0.6B-qtsa-mix-thinking",
    report_to="tensorboard",
    max_length=3000,
)

def formatting_func(ex):
    messages = [
        {"role": "user", "content": ex["instruction"]},
        {
            "role": "assistant",
            "content": ex["thinking"] + ex["response"]
        }
    ]

    # thinking mode enabled!!!
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=True      # <<< ENABLE thinking mode
    )
    return text


# ====== Trainer ========
trainer = SFTTrainer(
    model=model,
    formatting_func=formatting_func,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

# ====== Save model ========
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)


In [ ]:
# Fine-tuning Qwen-0.6B(non-thinking mode)

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# ==== local base model ====
base_model_path = "Qwen-0.6B-local"    
data_path = {
    "train": "validation_QTSA_mix/QTSA_train_sft.jsonl",
    "validation": "validation_QTSA_mix/QTSA_val_sft.jsonl"
}
output_dir = "outputs/qwen-0.6B-qtsa-mix-nothinking"

# ==== tokenizer ====
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

max_memory = {
    0: "30GiB",
    1: "30GiB",
}

# ==== load model ====
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    max_memory=max_memory,
    torch_dtype=torch.float16,
    use_cache=False,
    trust_remote_code=True,
)

base_model.gradient_checkpointing_enable()

# ==== LoRA ====
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

# ==== dataset ====
dataset = load_dataset("json", data_files=data_path)
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

# ==== SFT config ====
sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=50,
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir="logs/qwen-0.6B-qtsa-mix-nothinking",
    report_to="tensorboard",
    max_length=3000,
)

def formatting_func(ex):

    messages = [
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["response"]}
    ]

    # non-thinking mode
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False     # <<< DISABLE thinking mode
    )
    return text

# ==== trainer ====
trainer = SFTTrainer(
    model=base_model,
    formatting_func=formatting_func,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()

# ==== save ====
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
